
# COMET Evaluation for Final Analysis (Combined Inputs (Apples-to-Apples Idioms Test))

This notebook scores both:
- a **wide comparison CSV** like `preds_5models.csv`
- **per-model prediction CSVs** like the FLAN-T5 outputs

and produces combined outputs where `comet_summary_overall.csv` includes **all models**, including FLAN-T5.



## 0. Install note

Run installs in a fresh Colab runtime if needed, then restart the runtime manually.


In [1]:

# Optional install cell for a fresh Colab runtime.
# Uncomment and run only if COMET is not installed, then restart runtime manually.
!pip -q install -U unbabel-comet
# print("Restart the runtime now, then continue.")


In [2]:

from comet import download_model, load_from_checkpoint
import torch
import os
from pathlib import Path
import pandas as pd
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cpu


## 1. Paths and configuration

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

# Drive / project paths
PROJECT_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
QUAL_PREDS_DIR = os.path.join(PROJECT_ROOT, "qual_preds")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

OUT_DIR = os.path.join(QUAL_PREDS_DIR, "comet_eval")
os.makedirs(OUT_DIR, exist_ok=True)

# Input mode:
#   "wide_csv"       -> only the wide comparison CSV
#   "per_model_csvs" -> only the per-model CSVs
#   "both"           -> combine both sources in one scoring pass
INPUT_MODE = "both"

# Wide CSV input (from 20_preds notebook)
WIDE_CSV_PATH = os.path.join(QUAL_PREDS_DIR, "preds_5models.csv")

# Per-model inputs (from FLAN-T5 prompting notebook)
MODEL_FILES = {
    "flan_t5_small_prompt_0shot": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_0shot_preds.csv"),
    "flan_t5_small_prompt_3shot": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_3shot_preds.csv"),
    # "flan_t5_small_prompt_5shot": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_5shot_preds.csv"),
}

# Optional metrics merge
METRICS_FILES = [
    os.path.join(RESULTS_DIR, "metrics.csv"),
    os.path.join(RESULTS_DIR, "prompting_metrics.csv"),
    os.path.join(RESULTS_DIR, "prompting", "prompting_metrics.csv"),
]

# COMET model
COMET_MODEL_NAME = "Unbabel/wmt22-comet-da"
BATCH_SIZE = 8

# Apples-to-apples evaluation controls
# We define the canonical evaluation set from the wide CSV idioms subset
# and then filter FLAN outputs down to those exact same (src, ref) pairs.
CANONICAL_GROUP = "idioms_test"
EXPECTED_CANONICAL_SIZE = 25
APPLY_APPLES_TO_APPLES_FILTER = True
STRICT_CANONICAL_SIZE = False  # set True if you want the notebook to error when canonical size != EXPECTED_CANONICAL_SIZE


## 2. Helpers

In [5]:

def existing_path(p):
    return p is not None and os.path.exists(p)

def infer_group_from_row(row):
    for key in ("group", "split", "dataset", "eval_group"):
        if key in row and pd.notna(row[key]):
            return str(row[key])
    return "unknown"

def normalize_text_value(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).strip().split())

def normalize_wide_df(df):
    # source/reference columns
    src_col = None
    ref_col = None
    for c in ["src", "source", "english", "input"]:
        if c in df.columns:
            src_col = c
            break
    for c in ["tgt", "reference", "ref", "german", "target"]:
        if c in df.columns:
            ref_col = c
            break
    if src_col is None or ref_col is None:
        raise ValueError(f"Could not infer src/ref columns from wide CSV columns: {list(df.columns)}")

    pred_cols = [c for c in df.columns if c.startswith("pred_")]
    if not pred_cols:
        raise ValueError("No pred_* columns found in wide CSV.")

    rows = []
    for row_idx, r in df.iterrows():
        src = r[src_col]
        ref = r[ref_col]
        group = infer_group_from_row(r)
        example_id = r["id"] if "id" in df.columns else row_idx
        for pc in pred_cols:
            pred = r[pc]
            model = pc.replace("pred_", "")
            rows.append({
                "example_id": example_id,
                "src": src,
                "ref": ref,
                "pred": pred,
                "model": model,
                "group": group,
                "input_source": "wide_csv",
            })
    out = pd.DataFrame(rows)
    return out

def normalize_per_model_df(df, model_name):
    src_col = None
    ref_col = None
    pred_col = None

    for c in ["src", "source", "english", "input"]:
        if c in df.columns:
            src_col = c
            break
    for c in ["tgt", "reference", "ref", "german", "target"]:
        if c in df.columns:
            ref_col = c
            break
    for c in ["pred", "prediction", "mt"]:
        if c in df.columns:
            pred_col = c
            break

    if src_col is None or ref_col is None or pred_col is None:
        raise ValueError(f"Could not infer src/ref/pred columns for {model_name} from columns: {list(df.columns)}")

    for c in ["group", "split", "dataset", "eval_group"]:
        if c in df.columns:
            group_series = df[c].fillna("unknown").astype(str)
            break
    else:
        # FLAN prompting files are assumed to be idiom evaluation outputs
        group_series = pd.Series([CANONICAL_GROUP] * len(df), index=df.index)

    example_ids = df["id"] if "id" in df.columns else df.index

    out = pd.DataFrame({
        "example_id": example_ids,
        "src": df[src_col],
        "ref": df[ref_col],
        "pred": df[pred_col],
        "model": model_name,
        "group": group_series,
        "input_source": "per_model_csvs",
    })
    return out

def build_canonical_pairs_from_wide(wide_long_df, canonical_group=CANONICAL_GROUP):
    subset = (
        wide_long_df.loc[wide_long_df["group"] == canonical_group, ["example_id", "src", "ref"]]
        .drop_duplicates()
        .copy()
    )
    subset["src_norm"] = subset["src"].map(normalize_text_value)
    subset["ref_norm"] = subset["ref"].map(normalize_text_value)

    canonical_pairs = set(zip(subset["src_norm"], subset["ref_norm"]))
    print(f"Canonical group '{canonical_group}' unique examples:", len(subset))
    if STRICT_CANONICAL_SIZE and len(subset) != EXPECTED_CANONICAL_SIZE:
        raise ValueError(
            f"Canonical set size was {len(subset)} but EXPECTED_CANONICAL_SIZE={EXPECTED_CANONICAL_SIZE}. "
            "Set STRICT_CANONICAL_SIZE=False to allow this."
        )
    return subset, canonical_pairs

def filter_to_canonical(df, canonical_pairs, canonical_group=CANONICAL_GROUP):
    x = df.copy()
    x["src_norm"] = x["src"].map(normalize_text_value)
    x["ref_norm"] = x["ref"].map(normalize_text_value)
    x["pair_key"] = list(zip(x["src_norm"], x["ref_norm"]))

    # keep only rows that match the canonical pair set
    x = x[x["pair_key"].isin(canonical_pairs)].copy()

    # enforce a single comparable group label
    x["group"] = canonical_group
    return x

def validate_model_coverage(filtered_df, expected_n=EXPECTED_CANONICAL_SIZE):
    coverage = filtered_df.groupby("model").size().sort_values(ascending=False)
    print("Rows per model after apples-to-apples filtering:")
    print(coverage.to_string())
    return coverage

def load_inputs():
    parts = []
    canonical_subset = None
    canonical_pairs = None

    # Always load wide CSV first if it is part of the run so we can define the canonical set from it.
    if INPUT_MODE in ("wide_csv", "both"):
        if not existing_path(WIDE_CSV_PATH):
            raise FileNotFoundError(f"Wide CSV not found: {WIDE_CSV_PATH}")
        wide_df = pd.read_csv(WIDE_CSV_PATH)
        wide_long = normalize_wide_df(wide_df)
        print("Loaded wide CSV:", WIDE_CSV_PATH)

        if APPLY_APPLES_TO_APPLES_FILTER:
            canonical_subset, canonical_pairs = build_canonical_pairs_from_wide(wide_long, CANONICAL_GROUP)
            wide_long = filter_to_canonical(wide_long, canonical_pairs, CANONICAL_GROUP)
            print("Wide CSV rows after apples-to-apples filter:", len(wide_long))
        parts.append(wide_long)

    if INPUT_MODE in ("per_model_csvs", "both"):
        model_parts = []
        for model_name, path in MODEL_FILES.items():
            if not existing_path(path):
                print(f"Skipping missing per-model file for {model_name}: {path}")
                continue
            df = pd.read_csv(path)
            model_df = normalize_per_model_df(df, model_name)

            if APPLY_APPLES_TO_APPLES_FILTER:
                if canonical_pairs is None:
                    raise ValueError(
                        "Apples-to-apples filtering requires a canonical set from the wide CSV. "
                        "Use INPUT_MODE='both' or disable APPLY_APPLES_TO_APPLES_FILTER."
                    )
                before_n = len(model_df)
                model_df = filter_to_canonical(model_df, canonical_pairs, CANONICAL_GROUP)
                print(f"Loaded per-model CSV: {path} | kept {len(model_df)} of {before_n} rows after apples-to-apples filter")
            else:
                print("Loaded per-model CSV:", path)

            model_parts.append(model_df)

        if model_parts:
            parts.append(pd.concat(model_parts, ignore_index=True))

    if not parts:
        raise ValueError("No input data loaded. Check INPUT_MODE and file paths.")

    combined = pd.concat(parts, ignore_index=True)

    # drop fully empty predictions / refs
    combined["src"] = combined["src"].astype(str)
    combined["ref"] = combined["ref"].astype(str)
    combined["pred"] = combined["pred"].fillna("").astype(str)
    combined["group"] = combined["group"].fillna("unknown").astype(str)

    if APPLY_APPLES_TO_APPLES_FILTER:
        validate_model_coverage(combined, EXPECTED_CANONICAL_SIZE)

    return combined

def score_with_comet(df, model, batch_size=8):
    data = [
        {"src": s, "mt": p, "ref": r}
        for s, p, r in zip(df["src"].tolist(), df["pred"].tolist(), df["ref"].tolist())
    ]
    outputs = model.predict(data, batch_size=batch_size, gpus=1 if device=="cuda" else 0)
    scores = outputs.scores if hasattr(outputs, "scores") else outputs["scores"]
    out = df.copy()
    out["comet"] = scores
    return out

def summarize_scores(scored_df):
    overall = (
        scored_df.groupby("model", as_index=False)
        .agg(
            n=("comet", "size"),
            comet_mean=("comet", "mean"),
            comet_std=("comet", "std"),
            input_sources=("input_source", lambda s: ",".join(sorted(set(map(str, s)))))
        )
        .sort_values("comet_mean", ascending=False)
        .reset_index(drop=True)
    )

    by_group = (
        scored_df.groupby(["model", "group"], as_index=False)
        .agg(
            n=("comet", "size"),
            comet_mean=("comet", "mean"),
            comet_std=("comet", "std"),
            input_sources=("input_source", lambda s: ",".join(sorted(set(map(str, s)))))
        )
        .sort_values(["group", "comet_mean"], ascending=[True, False])
        .reset_index(drop=True)
    )

    by_input_source = (
        scored_df.groupby(["input_source", "model"], as_index=False)
        .agg(
            n=("comet", "size"),
            comet_mean=("comet", "mean"),
            comet_std=("comet", "std"),
        )
        .sort_values(["input_source", "comet_mean"], ascending=[True, False])
        .reset_index(drop=True)
    )

    pivot_by_group = by_group.pivot(index="model", columns="group", values="comet_mean").reset_index()
    pivot_by_input_source = by_input_source.pivot(index="model", columns="input_source", values="comet_mean").reset_index()

    return overall, by_group, by_input_source, pivot_by_group, pivot_by_input_source

def try_merge_metrics(overall_df):
    metric_frames = []
    for p in METRICS_FILES:
        if existing_path(p):
            try:
                x = pd.read_csv(p)
                x["metrics_source_file"] = os.path.basename(p)
                metric_frames.append(x)
            except Exception as e:
                print(f"Could not read metrics file {p}: {e}")

    if not metric_frames:
        print("No metrics files found for optional merge.")
        return None

    metrics = pd.concat(metric_frames, ignore_index=True, sort=False)

    # heuristics for name columns
    overall = overall_df.copy()
    left_key = "model"
    if "run_name" in metrics.columns:
        right_key = "run_name"
    elif "model_name" in metrics.columns:
        right_key = "model_name"
    else:
        print("Metrics files do not contain run_name/model_name. Skipping merge.")
        return None

    merged = overall.merge(metrics, left_on=left_key, right_on=right_key, how="left")
    return merged


## 3. Load COMET model

In [6]:

model_path = download_model(COMET_MODEL_NAME)
comet_model = load_from_checkpoint(model_path)
print("Loaded COMET model:", COMET_MODEL_NAME)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

Loaded COMET model: Unbabel/wmt22-comet-da


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


## 4. Load and combine inputs (apples-to-apples idioms_test subset)

In [7]:

combined_df = load_inputs()
print("Rows loaded:", len(combined_df))
print("Models:", sorted(combined_df["model"].unique().tolist()))
print("Input sources:", combined_df["input_source"].value_counts().to_dict())
print("Groups:", combined_df["group"].value_counts().to_dict())
combined_df.head()


Loaded wide CSV: /content/drive/MyDrive/ds266_idiom_mt/qual_preds/preds_5models.csv
Canonical group 'idioms_test' unique examples: 25
Wide CSV rows after apples-to-apples filter: 125
Loaded per-model CSV: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_0shot_preds.csv | kept 25 of 100 rows after apples-to-apples filter
Loaded per-model CSV: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_3shot_preds.csv | kept 25 of 100 rows after apples-to-apples filter
Rows per model after apples-to-apples filtering:
model
baseline                      25
flan_t5_small_prompt_0shot    25
flan_t5_small_prompt_3shot    25
idiom_only_v1                 25
lora_r16_stage2               25
lora_r4_stage2                25
lora_r8_stage2                25
Rows loaded: 175
Models: ['baseline', 'flan_t5_small_prompt_0shot', 'flan_t5_small_prompt_3shot', 'idiom_only_v1', 'lora_r16_stage2', 'lora_r4_stage2', 'lora_r8_stage2']
Input sources: {'wide_csv': 125, 'per_model_csv

,example_id,src,ref,pred,model,group,input_source,src_norm,ref_norm,pair_key
0,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",baseline,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu..."
1,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Papa sagt, ich bin nicht für das Militär gesch...",idiom_only_v1,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu..."
2,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r4_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu..."
3,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r8_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu..."
4,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r16_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu..."


## 5. Score with COMET

In [8]:

scored_df = score_with_comet(combined_df, comet_model, batch_size=BATCH_SIZE)
scored_df.head()


INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 22/22 [02:53<00:00,  7.89s/it]


,example_id,src,ref,pred,model,group,input_source,src_norm,ref_norm,pair_key,comet
0,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",baseline,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299
1,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Papa sagt, ich bin nicht für das Militär gesch...",idiom_only_v1,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.928583
2,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r4_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299
3,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r8_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299
4,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r16_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299


## 6. Summaries

In [9]:

overall_df, by_group_df, by_input_source_df, pivot_by_group_df, pivot_by_input_source_df = summarize_scores(scored_df)

print("Overall summary")
display(overall_df)

print("Summary by group")
display(by_group_df)

print("Summary by input source")
display(by_input_source_df)


Overall summary


,model,n,comet_mean,comet_std,input_sources
0,idiom_only_v1,25,0.762717,0.133694,wide_csv
1,lora_r16_stage2,25,0.732651,0.143304,wide_csv
2,baseline,25,0.732551,0.135192,wide_csv
3,lora_r8_stage2,25,0.732034,0.147128,wide_csv
4,lora_r4_stage2,25,0.727875,0.151279,wide_csv
5,flan_t5_small_prompt_0shot,25,0.381018,0.118532,per_model_csvs
6,flan_t5_small_prompt_3shot,25,0.363405,0.106758,per_model_csvs


Summary by group


,model,group,n,comet_mean,comet_std,input_sources
0,idiom_only_v1,idioms_test,25,0.762717,0.133694,wide_csv
1,lora_r16_stage2,idioms_test,25,0.732651,0.143304,wide_csv
2,baseline,idioms_test,25,0.732551,0.135192,wide_csv
3,lora_r8_stage2,idioms_test,25,0.732034,0.147128,wide_csv
4,lora_r4_stage2,idioms_test,25,0.727875,0.151279,wide_csv
5,flan_t5_small_prompt_0shot,idioms_test,25,0.381018,0.118532,per_model_csvs
6,flan_t5_small_prompt_3shot,idioms_test,25,0.363405,0.106758,per_model_csvs


Summary by input source


,input_source,model,n,comet_mean,comet_std
0,per_model_csvs,flan_t5_small_prompt_0shot,25,0.381018,0.118532
1,per_model_csvs,flan_t5_small_prompt_3shot,25,0.363405,0.106758
2,wide_csv,idiom_only_v1,25,0.762717,0.133694
3,wide_csv,lora_r16_stage2,25,0.732651,0.143304
4,wide_csv,baseline,25,0.732551,0.135192
5,wide_csv,lora_r8_stage2,25,0.732034,0.147128
6,wide_csv,lora_r4_stage2,25,0.727875,0.151279


## 7. Sentence spread (useful for qualitative analysis)

In [10]:

spread_df = (
    scored_df.pivot_table(
        index=["src", "ref", "group"],
        columns="model",
        values="comet",
        aggfunc="first",
    )
    .reset_index()
)

model_cols = [c for c in spread_df.columns if c not in ["src", "ref", "group"]]
if model_cols:
    spread_df["comet_max"] = spread_df[model_cols].max(axis=1)
    spread_df["comet_min"] = spread_df[model_cols].min(axis=1)
    spread_df["comet_spread"] = spread_df["comet_max"] - spread_df["comet_min"]
    spread_df = spread_df.sort_values("comet_spread", ascending=False).reset_index(drop=True)

spread_df.head(20)


model,src,ref,group,baseline,flan_t5_small_prompt_0shot,flan_t5_small_prompt_3shot,idiom_only_v1,lora_r16_stage2,lora_r4_stage2,lora_r8_stage2,comet_max,comet_min,comet_spread
0,But before she could launch her charm offensiv...,Doch bevor sie ihre Charmeoffensive starten ko...,idioms_test,0.932659,0.252559,0.169645,0.863792,0.932659,0.945956,0.932659,0.945956,0.169645,0.776311
1,"For the young Canadian servicemen, the order t...",Die jungen kanadischen Soldaten erwarteten täg...,idioms_test,0.812785,0.180693,0.199865,0.681498,0.812785,0.812785,0.812785,0.812785,0.180693,0.632092
2,"Bend the knee, peasant! Admit that I am your r...","Beuge das Knie, Bauer! Gib zu, dass ich dein r...",idioms_test,0.799448,0.347663,0.319701,0.917172,0.799448,0.799448,0.799448,0.917172,0.319701,0.597471
3,"""Regardless of their cause, inequalities are a...","""Unabhängig von ihrer Ursache sind Ungleichhei...",idioms_test,0.842750,0.290774,0.307880,0.876986,0.827501,0.860689,0.827501,0.876986,0.290774,0.586211
4,We're cut from the same cloth as any American ...,Wir sind aus dem gleichen Holz geschnitzt wie ...,idioms_test,0.728234,0.336061,0.336061,0.907209,0.848501,0.848501,0.848501,0.907209,0.336061,0.571148
5,"I wouldn't say they're like chalk and cheese, ...","Ich würde nicht sagen, dass sie wie Tag und Na...",idioms_test,0.775548,0.469741,0.241811,0.775548,0.775548,0.775548,0.775548,0.775548,0.241811,0.533737
6,"It would certainly be fun in an active family,...",Es wäre sicher lustig in einer aktiven Familie...,idioms_test,0.901928,0.376942,0.416186,0.909396,0.898602,0.900598,0.898602,0.909396,0.376942,0.532454
7,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...",idioms_test,0.924299,0.423057,0.404513,0.928583,0.924299,0.924299,0.924299,0.928583,0.404513,0.524070
8,Despite several bilateral and multilateral mee...,Trotz mehrerer bilateraler und multilateraler ...,idioms_test,0.866841,0.404440,0.415179,0.895262,0.867457,0.866841,0.867457,0.895262,0.404440,0.490822
9,"I don't want to put the cart before the horse,...",Ich möchte das Pferd nicht von hinten aufzäume...,idioms_test,0.743966,0.288209,0.316076,0.757543,0.744190,0.744190,0.744190,0.757543,0.288209,0.469333


## 8. Optional merge with BLEU / chrF

In [11]:

merged_df = try_merge_metrics(overall_df)
if merged_df is not None:
    display(merged_df.head())
else:
    print("No merged metrics dataframe created.")


,model,n,comet_mean,comet_std,input_sources,timestamp,run_name,split,bleu,chrf,...,max_new_tokens,freeze_encoder,train_size,dev_size,seed,n_eval,notes,metrics_source_file,n_shots,gen_kwargs
0,idiom_only_v1,25,0.762717,0.133694,wide_csv,2026-02-19T19:12:29-08:00,idiom_only_v1,idioms_test,44.363297,64.640647,...,128.0,False,800.0,100.0,42.0,100.0,stage1 idiom-only fine-tune,metrics.csv,NaN,NaN
1,idiom_only_v1,25,0.762717,0.133694,wide_csv,2026-02-19T19:14:16-08:00,idiom_only_v1,wmt_test,26.589856,58.217984,...,128.0,False,20000.0,2000.0,42.0,3003.0,stage1 idiom-only fine-tune (eval on WMT),metrics.csv,NaN,NaN
2,idiom_only_v1,25,0.762717,0.133694,wide_csv,2026-03-07T20:04:44-08:00,idiom_only_v1,idioms_test,44.363297,64.640647,...,128.0,False,800.0,100.0,42.0,100.0,stage1 idiom-only fine-tune,metrics.csv,NaN,NaN
3,idiom_only_v1,25,0.762717,0.133694,wide_csv,2026-03-07T20:06:28-08:00,idiom_only_v1,wmt_test,26.589856,58.217984,...,128.0,False,20000.0,2000.0,42.0,3003.0,stage1 idiom-only fine-tune (eval on WMT),metrics.csv,NaN,NaN
4,idiom_only_v1,25,0.762717,0.133694,wide_csv,2026-03-22T16:14:52-07:00,idiom_only_v1,idioms_test,44.363297,64.640647,...,128.0,False,800.0,100.0,42.0,100.0,stage1 idiom-only fine-tune,metrics.csv,NaN,NaN


## 9. Save outputs

In [12]:

sentence_scores_path = os.path.join(OUT_DIR, "comet_sentence_scores.csv")
sentence_spread_path = os.path.join(OUT_DIR, "comet_sentence_spread.csv")
overall_path = os.path.join(OUT_DIR, "comet_summary_overall.csv")
by_group_path = os.path.join(OUT_DIR, "comet_summary_by_group.csv")
pivot_group_path = os.path.join(OUT_DIR, "comet_pivot_by_group.csv")
by_input_source_path = os.path.join(OUT_DIR, "comet_summary_by_input_source.csv")
pivot_input_source_path = os.path.join(OUT_DIR, "comet_pivot_by_input_source.csv")
merged_path = os.path.join(OUT_DIR, "comet_merged_with_bleu_chrf.csv")

scored_df.to_csv(sentence_scores_path, index=False)
spread_df.to_csv(sentence_spread_path, index=False)
overall_df.to_csv(overall_path, index=False)
by_group_df.to_csv(by_group_path, index=False)
pivot_by_group_df.to_csv(pivot_group_path, index=False)
by_input_source_df.to_csv(by_input_source_path, index=False)
pivot_by_input_source_df.to_csv(pivot_input_source_path, index=False)
if merged_df is not None:
    merged_df.to_csv(merged_path, index=False)

print("Saved:")
for p in [
    sentence_scores_path, sentence_spread_path, overall_path, by_group_path,
    pivot_group_path, by_input_source_path, pivot_input_source_path
]:
    print(" -", p)
if merged_df is not None:
    print(" -", merged_path)


Saved:
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_sentence_scores.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_sentence_spread.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_summary_overall.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_summary_by_group.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_pivot_by_group.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_summary_by_input_source.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_pivot_by_input_source.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_merged_with_bleu_chrf.csv
